### 🧊 实验 4 —— Frozen Lake 的价值迭代与策略迭代

在本实验中，我们将继续基于简化的 **FrozenLake** 环境，深入探索马尔可夫决策过程（MDP）。在实验 3 的基础上，我们将重点学习两个关键方法：

1. **最优策略求解（价值迭代）**  
   我们将使用价值迭代算法来计算最优状态价值函数，并从中提取能够最大化长期回报的最优策略。

2. **最优策略求解（策略迭代）**  
   我们将使用策略迭代算法，该方法通过在“策略评估”和“策略改进”之间交替进行，直到收敛到最优策略。

In [ ]:
import gymnasium as gym
import numpy as np
np.set_printoptions(precision=2, suppress=True)
import random
from gymnasium.envs.toy_text.frozen_lake import generate_random_map

In [ ]:
# Define a smaller 3x3 map
DESC_3x3 = [
    "SFF",
    "FHF",
    "FFG",
]
env = gym.make("FrozenLake-v1",desc=DESC_3x3,is_slippery=True, render_mode="ansi")
obs, info = env.reset(seed=42)
print(env.render())

In [ ]:
# FrozenLake 风格地图的通用状态转移构造器（适用于任意矩形地图）
# 兼容 Gymnasium 的动作顺序：LEFT=0，DOWN=1，RIGHT=2，UP=3。

from typing import List, Tuple
import numpy as np

LEFT, DOWN, RIGHT, UP = 0, 1, 2, 3
ACTIONS = [LEFT, DOWN, RIGHT, UP]
DIRS = {
    LEFT:  (0, -1),
    DOWN:  (1, 0),
    RIGHT: (0, 1),
    UP:    (-1, 0),
}

def _grid_to_idx(r: int, c: int, ncols: int) -> int:
    return r * ncols + c

def _idx_to_grid(s: int, ncols: int) -> Tuple[int, int]:
    return divmod(s, ncols)

def _clip_move(r: int, c: int, dr: int, dc: int, nrows: int, ncols: int) -> Tuple[int, int]:
    rr, cc = r + dr, c + dc
    rr = min(max(rr, 0), nrows - 1)
    cc = min(max(cc, 0), ncols - 1)
    return rr, cc

def build_frozenlake_transitions(desc: List[str], is_slippery: bool = True):
    """
    Build transition probabilities for a FrozenLake-like grid.

    Args:
        desc: list of strings (rows), made of {'S','F','H','G'}.
        is_slippery: if True -> stochastic: {left, forward, right} each with prob 1/3.
                     if False -> deterministic in the intended direction.

    Returns:
        P: np.ndarray (S, A, S), transition probabilities
        R: np.ndarray (S, A, S), rewards (1 on entering 'G', else 0)
        absorbing: np.ndarray (S,), True for 'H' or 'G' (absorbing/self-loop)
        shape_2d: (nrows, ncols)
        flatten_map: np.ndarray (S,), identity map (r,c)->s ordering
    """
    nrows = len(desc)
    ncols = len(desc[0])
    S = nrows * ncols
    A = 4

    grid = np.array([list(row) for row in desc])
    is_hole = (grid == 'H')
    is_goal = (grid == 'G')
    absorbing = (is_hole | is_goal).reshape(-1)

    P = np.zeros((S, A, S), dtype=float)
    R = np.zeros((S, A, S), dtype=float)

    def step_from_state(s: int, a: int) -> int:
        r, c = _idx_to_grid(s, ncols)
        dr, dc = DIRS[a]
        rr, cc = _clip_move(r, c, dr, dc, nrows, ncols)
        return _grid_to_idx(rr, cc, ncols)

    for s in range(S):
        if absorbing[s]:
            # Absorbing states self-loop for all actions
            for a in ACTIONS:
                P[s, a, s] = 1.0
            continue

        for a in ACTIONS:
            if is_slippery:
                left = (a - 1) % 4
                right = (a + 1) % 4
                for aa in [left, a, right]:
                    s2 = step_from_state(s, aa)
                    P[s, a, s2] += 1.0/3.0
            else:
                s2 = step_from_state(s, a)
                P[s, a, s2] = 1.0

            # Reward for ARRIVING at goal
            for s2 in range(S):
                if P[s, a, s2] > 0:
                    rr, cc = _idx_to_grid(s2, ncols)
                    if grid[rr, cc] == 'G':
                        R[s, a, s2] = 1.0

    flatten_map = np.arange(S, dtype=int)
    return P, R, absorbing, (nrows, ncols), flatten_map

In [ ]:
P, R, absorbing, shape2d, flatmap = build_frozenlake_transitions(DESC_3x3, is_slippery=True)

In [ ]:
# Per-action SxS matrices
T_per_action = [P[:, a, :] for a in range(4)] 
P_all = np.array([T_per_action[0], T_per_action[1], T_per_action[2], T_per_action[3]])

In [ ]:
POLICY = [
    1,  # state 0 (top-left)
    2,  # state 1
    1,  # state 2
    1,  # state 3
    1,  # state 4 (center, possibly hole)
    1,  # state 5
    2,  # state 6
    2,  # state 7
    2,  # state 8 (goal state)
]
n_states = len(POLICY)
P_pi = np.zeros((n_states, n_states))
for s in range(n_states):
    a = POLICY[s]             
    P_pi[s, :] = P_all[a, s]   

Reward = [
0,  # state 0 (top-left)
0,  # state 1
0,  # state 2
0,  # state 3
0,  # state 4 (center, possibly hole)
0,  # state 5
0,  # state 6
0,  # state 7
1,  # state 8 (goal state)
]
Reward = np.array(Reward)
gamma = 0.9

## Part 1: 求解最优策略（价值迭代）

到目前为止，我们关注的都是**策略评估**，也就是在给定策略下估计其价值函数。在这一部分中，我们将进一步进入**控制**问题，也就是寻找能够最大化长期回报的**最优策略**。

核心思想是使用**价值迭代（value iteration）**。它将“策略评估”和“策略改进”合并为同一个更新公式：

$$
V_{k+1}(s) \;=\; \max_a \Big[ R(s) + \gamma \sum_{s'} P(s'|s,a) \, V_k(s') \Big].
$$

当价值函数收敛之后，我们就可以通过在每个状态下选择使上式取得最大值的动作，提取出最优策略：

$$
\pi^*(s) = \arg\max_a \Big[ R(s) + \gamma \sum_{s'} P(s'|s,a) V^*(s') \Big].
$$

### 步骤
1. 任意初始化价值函数 $V_0(s)$（例如全设为 0）。  
2. 按照“对动作取最大值”的规则，反复更新每个状态的价值。  
3. 当价值函数收敛时停止迭代（也就是每次更新带来的变化已经很小）。  
4. 在每个状态下选择对应的贪心动作，从而得到最优策略 $\pi^*$。  

### 需要完成的内容
- 使用你的状态转移模型 `P_all_prob` 和奖励函数，实现价值迭代。  
- 输出**最优状态价值**。  
- 输出**最优策略**，用动作编号表示（0 = LEFT，1 = DOWN，2 = RIGHT，3 = UP）。

In [ ]:
# Your time to work on it

## Part 2: 求解最优策略（截断策略迭代）

到目前为止，我们已经看到了两种“极端”方法：  
- **价值迭代（Value Iteration）**：在每一次更新时都直接对动作取最大值；  
- **策略迭代（Policy Iteration）**：先对当前策略进行充分评估，再进行策略改进。  

**截断策略迭代（Truncated / Modified Policy Iteration）** 则在两者之间取得平衡：在每次策略改进之前，只进行**少量几步**策略评估。

形式上，我们在**部分策略评估**和**策略改进**之间交替进行：

$$
V^{(j+1)}(s) \;=\; R(s) \;+\; \gamma \sum_{s'} P(s' \mid s, \pi_k(s))\, V^{(j)}(s'),
\quad j = 0,1,\dots,m-1,
$$

然后进行策略改进：

$$
\pi_{k+1}(s) \;=\; \arg\max_{a} \Big[ \, R(s) \;+\; \gamma \sum_{s'} P(s' \mid s, a)\, V^{(m)}(s') \, \Big].
$$

### 步骤
1. 初始化价值函数 $V_0(s)$（例如全设为 0）以及一个初始策略 $\pi_0$。  
2. **部分策略评估**：在固定策略 $\pi_k$ 的情况下，只进行 $m$ 次贝尔曼**期望更新**来更新价值函数 $V$。  
3. **策略改进**：基于更新后的价值函数 $V$，在每个状态下选择能够最大化期望回报的动作，从而更新策略。  
4. 重复步骤 2 和步骤 3，直到策略不再发生变化（或者价值函数 / 策略的变化已经小于某个很小的阈值）。  

### 需要完成的内容
- 使用你的状态转移模型 `P_all_prob` 和奖励函数 `R`，实现截断策略迭代。  
- 选择一个较小的评估步数 $m$（例如 1–3），**或者**在每轮策略评估中设置容差阈值 $\theta$，满足条件时提前停止。  
- 输出**最终状态价值**和**得到的策略**，用动作编号表示  
  `(0 = LEFT, 1 = DOWN, 2 = RIGHT, 3 = UP)`。  
- （可选）在你的地图上比较截断策略迭代与**完整策略迭代**、**价值迭代**在迭代次数或运行时间上的差异。

In [ ]:
# Your time to work on it

## 对比：复现收敛曲线（PI vs. VI vs. 截断 PI）

目标：绘制一张图，展示 **价值迭代（VI）**、**策略迭代（PI）** 和 **截断策略迭代（TPI）** 如何在同一个 MDP 上（例如你的 FrozenLake 地图）逐步收敛到 **最优状态价值** \(V^\*\)。

### 实验设置
- 使用与第 1–4 部分相同的**环境**、折扣因子 $\gamma$ 和奖励定义。
- 固定一个**代表性状态** $s_\text{ref}$（推荐使用起始状态 `S`）。  
  纵轴取为 $V_k(s_\text{ref})$。
- 对 VI 和 TPI 使用**同步更新**，以保证曲线更平滑（一次完整 sweep 记作 1 次迭代）。

### 绘图要求
- 横轴：迭代次数 $k$。
- 纵轴：$V_k(s_\text{ref})$。
- 绘制以下内容：
  - **价值迭代（Value Iteration）**：洋红色折线，菱形标记，对应 `vals_vi`。
  - **策略迭代（Policy Iteration）**：蓝色折线，圆形标记，对应 `vals_pi`。
  - **截断策略迭代（Truncated PI）**：黑色折线，方形标记，对应 `vals_tpi`。
  - **最优状态价值**：在 $y = V^*(s_\text{ref})$ 处画一条红色**水平线**，标签为 **Optimal state value**。
- 添加图例：
  - Policy iteration
  - Value iteration
  - Truncated policy iteration
  - Optimal state value
- 坐标轴标签：
  - 横轴：`k`（iterations）
  - 纵轴：`V`（state value）
- 添加浅色网格线，便于阅读。

In [ ]:
# Your time to work on it